### Statistic for P117

In [1]:
import astroquery
from astroquery.eso import Eso
import pandas as pd
import numpy as np
import sys
import math
from numpy import *
from astropy.table import Table
from astropy.table import *
from astropy.io import ascii
from datetime import datetime, date, timedelta
from termcolor import colored

eso=Eso()
eso.ROW_LIMIT = -1 


pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns',None)

Defines a general search for available data in the ESO archive
- using a date range, PIDs and instrument as input¶

In [2]:
def eso_query(instr,start,end,pid):
    num_obs= 0
    tot_exptime = 0.0
    table = eso.query_main(column_filters={'instrument':instr,'dp_cat': 'SCIENCE','stime':start,'etime':end,'prog_id':pid})
    if not table:
        num_Obs      = num_obs + 0
        tot_exptime  = 0.0
    else:
        num_Obs = num_obs + len(table)
        pd_table    = table.to_pandas()
        tot_exptime = pd_table["Exptime"].sum()
    
    return num_Obs,tot_exptime


def eso_query_target(instr,target,start,end,pid):
    num_obs= 0
    tot_exptime_ir = 0.0
    tot_exptime_ir = 0.0
    table = eso.query_main(column_filters={'instrument':instr,'object':target,'dp_cat': 'SCIENCE','stime':start,'etime':end,'prog_id':pid})
    if not table:
        num_Obs      = num_obs + 0
        tot_exptime  = 0.0
    else:
        num_Obs = num_obs + len(table)
        pd_table    = table.to_pandas()
        tot_exptime = pd_table["Exptime"].sum()
    
    return num_Obs,tot_exptime


def eso_query_grond(instr,start,end,pid):
    num_ir = 0
    num_opt=0
    num_obs = 0
    table = eso.query_main(column_filters={'instrument': instr, 'prog_id':pid,'dp_cat': 'SCIENCE','stime':start,'etime':end} )
        
    if not table: 
        num_opt = 0
        num_nir  = 0
        tot_exptime_ir = 0.0
        tot_exptime_opt = 0.0
        tot_time = 0.0
        
    else:
# Check for optical images        
        lambda_lo       = 1100
        table_opt       = table[table['filter_lambda_min'] < lambda_lo] # limit list to only optical files
        pd_table        = table_opt.to_pandas()
        num_opt         = len(table_opt)                                     # number of optical images
        tot_exptime_opt = (pd_table["Exptime"].sum())
        table_nir       = table[table['filter_lambda_min'] > lambda_lo]      # number of near-infrared images
        pd_table        = table_nir.to_pandas()
        tot_exptime_ir  = pd_table["Exptime"].sum()                        
        num_nir = len(table_nir)
        num_obs = num_opt + num_nir
#        print(pid_1, "   ",num_nir, "    ",tot_exptime_ir)
    return num_opt,num_nir,tot_exptime_ir, tot_exptime_opt


def eso_grond_ir(instr,start,end,pid):
    num         = 0
    tot_exptime = 0
    table = eso.query_main(column_filters={'instrument': instr, 'prog_id':pid,'dp_cat': 'SCIENCE','stime':start,'etime':end} )
        
    if not table: 
        num         = 0
        tot_exptime = 0.0
        tot_time    = 0.0
        
    else:
# Check for optical images        
        lambda_lo       = 1100
        
        table_nir       = table[table['filter_lambda_min'] > lambda_lo]      # number of near-infrared images
        pd_table        = table_nir.to_pandas()
        tot_exptime     = pd_table["Exptime"].sum()                        
        num             = len(table_nir)


    return num,tot_exptime

def eso_grond_opt(instr,start,end,pid):
    num         = 0
    tot_exptime = 0
    table = eso.query_main(column_filters={'instrument': instr, 'prog_id':pid,'dp_cat': 'SCIENCE','stime':start,'etime':end} )
        
    if not table: 
        num         = 0
        tot_exptime = 0.0
        tot_time    = 0.0
        
    else:
# Check for optical images        
        lambda_lo       = 1100
        table_opt       = table[table['filter_lambda_min'] < lambda_lo]      # number of near-infrared images
        pd_table        = table_opt.to_pandas()
        tot_exptime     = pd_table["Exptime"].sum()                        
        num = len(table_opt)

    return num,tot_exptime





#### Total exposure times for GROND are calculated based on number of IRACE exposures and IRACE exposure times¶

In [16]:
info = "/home/angela/LaSilla/P117/P117_pi.list"

data = open(info,'r')

#start   = input('Start date of query (yyyy-mm-dd):  ')
#end     = input('End date of query (yyyy-mm-dd):    ')
start_1 = "2026-05-01"
end_1 = (datetime.now()+timedelta(days=2)).strftime('%Y-%m-%d')             # date for tomorrow, to make sure that all nights are  used
end_title= (datetime.now()+timedelta(days=-1)).strftime('%Y-%m-%d')  


#text = colored(("Dome repair between: ",end_1, "and :", start_1), "red")

print("Statistics until: ",end_1, "to make sure all data are found")   

end_f   = (datetime.now() + timedelta(days=-1) ).strftime('%Y-%m-%d')     # date for yesterday, the day the last observing night started
#end_f   = end.strftime('%Y-%m-%d')

print("     ")
print("     ")
print("     ")

orig_stdout = sys.stdout

path = "/home/angela/LaSilla/P117"

file_out   = path + "/Statistic/P117_stats.cvs"
file_out_2 = path + "/Statistic/P117_stats_a.cvs"

data_2 = []
col1=[]    # full name
col2=[]    # list of PIDs
col3=[]    # instrument
col4=[]    # allocated time in hours Category A
col5=[]    #  requested time in proposal
col6=[]    # number of files
col7=[]    # exposure times
col8=[]    # execution time in hr
col9=[]    # fraction in %
col10 =[]  # fraction completed in of total %


data_3 = []
cola=[]    # full name
colb=[]    # 1st PID
colc=[]    # 2nd PID
cold=[]    # instrument
cole=[]    # allocated time A in hr
colf=[]    # tie asked for in proposal
colg=[]    # number of files
colh=[]    # exposure time in hr
coli=[]    # observed time in hr
colj=[]    # fraction completed in %
colk=[]    # fraction of total time asked for in proposal

num_DDT     = 0
num         = 0
tot_exptime = 0.0
header1 = data.readline()
#header2 = data.readline()

for line in data:    
    num = 0
    tot_exptime = 0.0
    tot_time    = 0.0
    
    line     = line.strip()
    columns  = line.split()
    name     = columns[0]
    first    = columns[1]
    email    = columns[2]
    pid_1    = columns[3]
    pid_2    = columns[4]
    instr    = columns[5]
#    t_tot_A  = float(columns[6])    
#    t_tot_B  = float(columns[7])
    t_tot_A  = int(float(columns[6]))    
    t_tot_B  = int(columns[7])    
    tot_alloc_time = t_tot_A  + t_tot_B 
#    print(name,tot_alloc_time)
    f_name = name+','+ first
    full_name = ("{:24}".format(f_name))

    idlist = pid_1
    if pid_2 != str("000.0000.000"):
        idlist = idlist+", "+pid_2


    
#############################################################################################
   
# Check FEROS programs
  
    if instr  == "FEROS":
        num         = 0
        tot_exptime = 0.0
        tot_time    = 0.0
# check before dome repair        
        result_1_a = eso_query(instr,start_1,end_1,pid_1)
        num = num+result_1_a[0]
        tot_exptime= tot_exptime+result_1_a[1]
        result_1_b = eso_query(instr,start_1,end_1,pid_2)
        num = num+result_1_b[0]
        tot_exptime= tot_exptime+result_1_b[1]

        
        oh_time = 282.0
        tot_exptime = tot_exptime/3600.0
        tot_time    = ((num*oh_time)/3600.0)  + tot_exptime

        if name== "KORHONEN":                                   
            num           = 0
            tot_exptime   = 0.0
            tot_time_     = 0.0          
            oh_time       = 282.0                        # mean overhead
            result_1_a   = eso_query(instr,start_1,end_1,pid_1)
            num          = num+result_1_a[0]
#            print("Number of files bevore dome repair: ", num     )
            tot_exptime  = (tot_exptime+result_1_a[1])/3600.0
            tot_time     = tot_exptime + (num*oh_time)/3600.0

        if name== "Vaughan":            ##     That is a DDT program                117.2AS7     
            num           = 0
            tot_exptime   = 0.0
            tot_time_     = 0.0          
            oh_time       = 122.0                        # mean overhead
            result_1_a   = eso_query(instr,start_1,end_1,pid_1)
            num          = num+result_1_a[0]
#            print("Number of files bevore dome repair: ", num     )
            tot_exptime  = (tot_exptime+result_1_a[1])/3600.0
            tot_time     = tot_exptime + (num*oh_time/3600.0)

        if name== "AHRER":                            # either 3 or 6 exposures with 122 or 94 sec overhed per image, use 110 sec as average
            num          = 0
            tot_exptime  = 0.0
            tot_time     = 0.0
            oh_time      = 110.0
            result_1_a   = eso_query(instr,start_1,end_1,pid_1)
            num          = num+result_1_a[0]          
            tot_exptime  = (tot_exptime+result_1_a[1])/3600.0
            tot_time     = tot_exptime   + ((num*110)/3600.0)


        if name== "COSTA":
            num= 0
            tot_exptime  = 0.0
            tot_time     = 0.0
            oh_time      = 324.0/2
            result_1_a   = eso_query(instr,start_1,end_1,pid_1)
            num          = num+result_1_a[0]          
            tot_exptime  = (tot_exptime+result_1_a[1])/3600.0
            tot_time     = tot_exptime   + ((num*110)/3600.0)



            
###########################################################################################       
    #    Check WFI programs
    if instr == "WFI" :
        oh_time      = 226.0            # new overhead, 3.5 min for fetch... + setup + readout
        num          = 0
        tot_time     = 0.0
        tot_exptime  = 0.0
        if name =="KORHONEN":
            num= 0
            tot_exptime  = 0
            tot_time     = 0.0
            target = "XX CHA"
            oh_time = 930.0
            result_1_a   = eso_query_target(instr,target,start_1,end_1,pid_1)
            num          = num+result_1_a[0]
            tot_exptime  = tot_exptime+result_1_a[1]
            result_1_b   = eso_query_target(instr,target,start_1,end_1,pid_2)
            num          = num+result_1_b[0]
            tot_exptime  = (tot_exptime+result_1_b[1]) / 3600.0
            tot_time     = (((num/6)*oh_time)/3600.0)  + tot_exptime
#           print("Number of files: ", num     )  
        
        if name =="BANADOS":
            num = 0
            tot_exptime  = 0
            tot_time     = 0.0
            target_1 = "2MASS J10253030+1402078"
            result_1_a   = eso_query_target(instr,target_1,start_1,end_1,pid_1)
            num          = num+result_1_a[0]
            tot_exptime  = tot_exptime+result_1_a[1]
            result_1_b   = eso_query_target(instr,target_1,start_1,end_1,pid_2)
            num          = num+result_1_b[0]
            tot_exptime  = (tot_exptime+result_1_b[1]) / 3600.0
            oh_time_1    = 220.0
            tot_time     = ((num*oh_time_1)/3600.0)  + tot_exptime
   
#            print("Number of files: ", num     )

        if name =="DEMIANENKO":
            num = 0
            tot_exptime  = 0
            tot_time     = 0.0
            target_1 = "2MASS J14485011+1608030"
            result_1_a   = eso_query_target(instr,target_1,start_1,end_1,pid_1)
            num          = num+result_1_a[0]
            tot_exptime  = (tot_exptime+result_1_a[1])/3600.0
            oh_time_1    = 91.0                         # overheads for ac
            tot_time     = ((num*oh_time_1)/3600.0)  + tot_exptime
   
#            print("Number of files: ", num     )

        if name =="SUYU":
            num = 0
            tot_exptime  = 0
            tot_time     = 0.0
            target_1 = "SNR2026NGR"
            result_1_a   = eso_query_target(instr,target_1,start_1,end_1,pid_2)
            num          = num+result_1_a[0]
            tot_exptime  = (tot_exptime+result_1_a[1])/3600.0
            oh_time_1    = 128.0                         # overheads for ac
            tot_time     = ((num*oh_time_1)/3600.0)  + tot_exptime
   
#            print("Number of files: ", num     )
                
###################################################################################
# Check  GROND   programms
# output of eso_query_grond: num_opt,num_nir,tot_exptime_ir, tot_exptime_opt
    
    if instr == "GROND":
        num_opt       = 0
        num_ir        = 0
        overhead      = 220.0                                              # overheads per OB are now 3.5 min, 210 sec
        tot_exptime   = 0.0                                                # exposure time in optical
        tot_time      = 0.0                                                # execution time#
        
            
        if name=="RAU":
            overhead    = 0.0             # overheads are minimal for the large number of IRACE images, set to 0.0
            num         = 0
            num_ir      = 0
            num_opt     = 0
            tot_exp_ir_1 = 0.0
            tot_exp_ir_2 = 0.0
            
            result_opt_1  = eso_grond_opt(instr,start_1,end_1,pid_1)
            num_opt_1     = num_opt + result_opt_1[0] # number of opt images, optical is not used
#            tot_exptime = (tot_exptime + result_opt[1])/3600.0
            result_opt_2  = eso_grond_opt(instr,start_1,end_1,pid_2)
            num_opt_2     = num_opt + result_opt_2[0] # number of opt images, optical is not used
#            tot_exptime = (tot_exptime + result_opt[1])/3600.0
            num_opt       = num_opt_1 +num_opt_2
            result_ir_1   = eso_grond_ir(instr,start_1,end_1,pid_1)      # PID 1
            num_ir        = num_ir + result_ir_1[0]
            tot_exp_ir_1  = tot_exp_ir_1+ 6* result_ir_1[1]                     

            result_ir_2   = eso_grond_ir(instr,start_1,end_1,pid_2)      # PID 2
            num_ir        = num_ir + result_ir_2[0]
            tot_exp_ir_2  = tot_exp_ir_2+ 6* result_ir_2[1]    

            num = num_ir + num_opt           
            tot_exptime   = (tot_exp_ir_1 + tot_exp_ir_2)/3600.0      
            tot_time      = tot_exptime +num*overhead                    # overheads for IRACE images 
#                                                                         (including the acquisition) are very small

                
###################################################################################


    
#### Calculate percentages of allocated times

    time_perc_A = 0.0                                                       # percentage of Cat A. time observed
    time_perc_B = 0.0                                                       # percentage of Cat B. time observed
    
# Case 1: only category A is assigned, and total execution time is greater than 0.0:
    
    if tot_time > 0.0 and t_tot_A > 0.0 and t_tot_B == 0.0:
        time_perc_A = (100.0* (tot_time /t_tot_A))
        time_perc_B = 0.0
    elif tot_time > 0.0 and t_tot_A > 0.0 and t_tot_B > 0.0 and tot_time < t_tot_A:   # time observed is less than Cat. A
        time_perc_A = (100.0* (tot_time /t_tot_A))
        

# Case 2: only category B is assigned, and total execution time is greater than 0.0:        
    elif  tot_time > 0.0 and t_tot_B > 0.0 and t_tot_A == 0.0:  
        time_perc_B = 100.0* (tot_time /t_tot_B)
        time_perc_A = 0.0

# Case 3: both category A & B are assigned, and the total execution time is greater than the time assigned to Category A:    
    elif tot_time > t_tot_A and t_tot_B > 0.0  and t_tot_A !=0: 
        time_perc_A = 100.0
        time_perc_B = (100.0* (tot_time /t_tot_A))-100.0

# Case 4: no time has been assigned to either category (not useful):    
    elif t_tot_A == 0 and t_tot_B == 0:
        time_perc_A = 0.0
        time_perc_B = 0.0

# Case 5: no observation has been done:
    elif tot_exptime  == 0.0:
        time_perc_A   = 0.0
        time_perc_B   = 0.0




### Format the values
    
    t_tot_A_f      = ("{:>6.0f}".format(t_tot_A))                            # formatted allocated time cat- A
    t_tot_B_f      = ("{:>6.0f}".format(t_tot_B))                            # formatted allocated time cat- B
    tot_time_f     = ("{:>6.2f}".format(tot_time))                           # formatted observed time
    time_perc_A_f  = ("{:>6.2f}".format(time_perc_A))
    time_perc_B_f  = ("{:>6.2f}".format(time_perc_B))
    tot_exptime_f  = ("{:>6.2f}".format(tot_exptime))
#    print(name,num, tot_time_f,tot_exptime_f)

    data_2.append({'#PI_name                ': full_name.ljust(20," "),
                   'PIDs              ': idlist.ljust(30," "),
                   'Inst.': str(instr).center(5," "),
                   'Cat-A [h]': str(t_tot_A_f).rjust(10," "),
                   'Cat-B [h]': str(t_tot_B_f).rjust(10," "),
                   '# files': str(num).rjust(10," "),
                   'exp. [h]': tot_exptime_f,
                   'exec. [h] ': str(tot_time_f).center(10," "),
                   '  A [%]': time_perc_A_f,
                   '  B [%]': time_perc_B_f
        }              
    )
    col1.append(full_name)
    col2.append(idlist.ljust(30," "))
    col3.append(instr.center(9," "))
    col4.append(t_tot_A_f)
    col5.append(t_tot_B_f)
    col6.append(num)
    col7.append(tot_exptime_f)
    col8.append(tot_time_f)
    col9.append(time_perc_A_f)
    col10.append(time_perc_B_f)
              
    data_3.append({'#PI_name': full_name.ljust(20," "),
                   '       PID_1       ': pid_1.center(15," "),
                   '       PID_2       ': pid_2.center(15," "),
                   'Instrument  ': str(instr).center(5," "),
                   'Cat-A [h]': str(t_tot_A_f).rjust(10," "),
                   'Cat-B [h]': str(t_tot_B_f).rjust(10," "),
                   '# of files': str(num).rjust(10," "),
                   'exp. [h]': tot_exptime_f,
                   'exec.[h] ': str(tot_time_f).center(10," "),
                   '  A [%]': time_perc_A_f,
                   '  B.[%]': time_perc_B_f
        }              
    )
    cola.append(full_name)
    colb.append(pid_1.center(15," "))
    colc.append(pid_2.center(15," "))
    cold.append(instr.center(15," "))
    cole.append(t_tot_A_f)
    colf.append(t_tot_B_f)
    colg.append(num)
    colh.append(tot_exptime_f)
    coli.append(tot_time_f)
    colj.append(time_perc_A_f)
    colk.append(time_perc_B_f)


pd.set_option('display.precision', 1)    
df = pd.DataFrame(data_2)
df_1 = pd.DataFrame(data_3)
number = len(df.index)
#print(number)
print(df.to_string(index=False))
#print(new)
li = [df.columns.values.tolist()] + df.values.tolist()
df.to_csv(file_out, quoting=None,index=False) 
df_1.to_csv(file_out_2, index=False) 
print("    ")
print("    ")

print("Finished")
sys.stdout=orig_stdout

Statistics until:  2026-07-03 to make sure all data are found
     
     
     


#PI_name                             PIDs               Inst.  Cat-A [h]  Cat-B [h]    # files exp. [h] exec. [h]    A [%]   B [%]
AHRER,Eva-Maria          117.2ASK.001                   FEROS        211          0         87    34.50    37.16     17.61    0.00
AHRER,Eva-Maria          117.2ASL.001                   FEROS         17          0          0     0.00     0.00      0.00    0.00
EL-BADRY,Kareem          117.2AS1.001                   FEROS        200          0         67    46.75    51.99     26.00    0.00
MÜLLER-HORN,Johanna      117.2ARA.001                   FEROS         99          0         29    28.67    30.94     31.25    0.00
MÜLLER-HORN,Johanna      117.2ARB.001                   FEROS         33          0         51    39.47    43.46    131.70    0.00
MÜLLER-HORN,Johanna      117.2ARC.001                   FEROS          0         75          0     0.00     0.00      0.00    0.00
BANADOS,Eduardo          117.2AS3.002                    WFI          92          0

### This next part writes a pdf file, containing a table with the completness


In [17]:
import reportlab
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter,A4
from reportlab.lib.units import inch
from reportlab.lib import colors
import os
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle,Paragraph,Spacer
from reportlab.lib.styles import ParagraphStyle,getSampleStyleSheet
from reportlab.lib.units import cm

styles = getSampleStyleSheet()

path = "/home/angela/LaSilla/P117/Statistic/"
date_now = (datetime.now()).strftime('%Y-%m-%d')
pdf_out = path + end_f+".pdf"
print(pdf_out)
doc = SimpleDocTemplate(pdf_out, pagesize=A4,fontsize=8)
# container for the 'Flowable' objects
elements = []



colw = (1.3*inch,1.2*inch,0.55*inch,0.6*inch, 0.6*inch, 0.6*inch, 0.6*inch,0.6*inch, 0.6*inch,0.6*inch)
t=Table(li,colw,(number+1)*[0.3*inch])

t.setStyle(TableStyle([('ALIGN',(1,1),(8,number),'CENTER'),
                       ('ALIGN',(1,1),(1,number),'LEFT'),
                       ('FONTSIZE',(0,0),(10,number),5),
                       ('TEXTCOLOR',(1,1),(10,number),colors.black),
                       ('TEXTCOLOR',(0,0),(0,-1),colors.blue),
                       ('BACKGROUND', (0, 0), (0, number), colors.lightblue),
                       ('BACKGROUND', (1, 0), (10, 0), colors.lightblue),
                       ('BACKGROUND',(0,number-4),(0,number-1),colors.orange),
                       ('BACKGROUND',(0,number),(0,number),colors.red),                      
                       ('VALIGN',(0,-1),(-1,-1),'MIDDLE'),
                       ('INNERGRID', (0,0), (-1,-1), 0.25, colors.black),
                       ('BOX', (0,0), (-1,-1), 0.25, colors.black),
                       ]))
elements.append(t)


title = "P117 from " + start_1 + " until "+end_title

flowables = [
    Paragraph(title, styles['Title']),
    t,
    Spacer(1 * cm, 1 * cm),
Paragraph('- I count the data for XX-CHA as separate program'),    
Paragraph('- Chile time is given in number of nights NOT hours, 1 night= 10 hours!!!'),
Paragraph('- GROND is difficult to count, they change their OBs for the same target')
]
doc.build(flowables)

/home/angela/LaSilla/P117/Statistic/2026-06-30.pdf


### If you want to test a single PID for WFI

In [ ]:
########WFI
pid_1        = "117.2AS3.002"            # 
pid_2        = "000.0000.000"
instr        = "WFI"
start_1      = "2026-04-27"
end_1        = (datetime.now()+timedelta(days=2)).strftime('%Y-%m-%d') 

oh_time      = 230.0            # new overhead, 3.5 min for fetch... + setup + readout
num          = 0
tot_time     = 0.0
tot_exptime  = 0.0

result_1_a   = eso_query(instr,start_1,end_1,pid_1)
num          = num+result_1_a[0]
tot_exptime  = tot_exptime+result_1_a[1]
result_1_b   = eso_query(instr,start_1,end_1,pid_2)
num          = num+result_1_b[0]

tot_exptime  = (tot_exptime+result_1_b[1]) / 3600.0
tot_time     = ((num*oh_time)/3600.0)  + tot_exptime

print("Total number of files: ", num     )
print("Total exposure time :", tot_exptime)



### or FEROS

In [ ]:
pid_1        = "117.2ARW.001"            #
instr        = "FEROS"
start_1      = "2026-05-01"
end_1        = (datetime.now()+timedelta(days=2)).strftime('%Y-%m-%d') 

num          = 0
tot_exptime  = 0.0
tot_time     = 0.0
oh_time      = 282.0


result_1_a   = eso_query(instr,start_1,end_1,pid_1)
num          = num+result_1_a[0]
print("Number of files for ", pid_1,": ",num     )
tot_exptime  = tot_exptime+result_1_a[1]
tot_time     = ((num*oh_time)  + tot_exptime) / 3600.0
print("Total exposure time (h):", tot_exptime/3600.0)


### or GROND
####  based on one selected target
####  Usually, there are 6 co-adds in one IRACE image

In [ ]:
pid_1        = "116.29GN.001"            #  
pid_2        = "000.0000.000"            #  
instr        = "GROND"
start_1      = "2026-06-01"
end_1        = (datetime.now()+timedelta(days=2)).strftime('%Y-%m-%d') 

num          = 0
num_opt      = 0
num_ir       = 0
tot_exptime  = 0.0
tot_time     = 0.0
overhead= 220.0
name         = "RAU"
target       = "SN2025UVO"

num         = 0
tot_exptime = 0
table = result_1    = eso_query_grond(instr,start_1,end_1,pid_1)
        
if not table: 
    num         = 0
    tot_exptime = 0.0
    tot_time    = 0.0
        
else:
# Check for optical images      
    result_ir_1   = eso_grond_ir(instr,start_1,end_1,pid_1)      # PID 1
    num_ir        = num_ir + result_ir_1[0]
    tot_exp_ir_1  = tot_exp_ir_1+ 6* result_ir_1[1]  
    result_opt_1  = eso_grond_opt(instr,start_1,end_1,pid_1)     # PID 1
    num_opt       = num_opt + result_grod_opt[0]
    tot_exp_opt_1 = tot_exp_opt+ result_opt_1[1]

    num           = num_ir + num_opt
    tot_exp_ir    = 6 * tot_exp_ir_1
    tot_time      = tot_exp_ir                                  # only total execution time based on IRACE is counted, overheads are set to 0.0


print("Total number of optical and near-infrared images:  num")
print("Total exposure time, based on IRACE: tot_exp_ir")





In [ ]:
print(num_ir)